# 04 — Customer Retention, Cohorts, and Segmentation

Phase 5: evaluates whether traditional RFM segmentation is appropriate for this
dataset, builds a censoring-aware cohort retention analysis, and implements a
defensible behavior-based customer segmentation using `src/segmentation.py`.
No dashboard styling here — this is the analytical layer for Phase 6.

**Population:** delivered orders only, keyed on `customer_unique_id` throughout
(never `customer_id` — see README.md, "Data Model").

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import plotly.express as px

from src import metrics as m
from src import segmentation as seg
from src.database import get_connection

con = get_connection()
orders_analytics = con.execute("SELECT * FROM orders_analytics").fetchdf()

customer_summary = seg.build_customer_summary(orders_analytics)
census_date = orders_analytics.loc[orders_analytics["is_delivered"], "order_purchase_timestamp"].max()
print(f"Census date (dataset's last observed delivered-order purchase): {census_date}")
print(f"Customers: {len(customer_summary):,}")

Census date (dataset's last observed delivered-order purchase): 2018-08-29 15:00:37
Customers: 93,358


## Customer Frequency, Recency, and Monetary Distributions (RFM Evaluation)

In [2]:
report = seg.build_rfm_suitability_report(customer_summary)

print(f"% of customers with Frequency = 1: {report['pct_frequency_equals_1']*100:.1f}%")
print()
print("Frequency distribution:")
print((report["frequency_value_counts"] * 100).round(2).head(10))

% of customers with Frequency = 1: 97.0%

Frequency distribution:
delivered_order_count
1     97.00
2      2.76
3      0.19
4      0.03
5      0.01
6      0.01
7      0.00
9      0.00
15     0.00
Name: proportion, dtype: float64


**Frequency is degenerate.** 97.0% of customers have exactly one delivered order. A quintile-based Frequency score would put roughly 97% of customers in the same bin — it wouldn't differentiate anyone, and any remaining "spread" would just be noise from the tiny 3% tail. This alone is enough to disqualify a standard 5x5x5 RFM scheme.

In [3]:
fig = px.histogram(customer_summary, x="recency_days", nbins=60,
                    title="Recency distribution (days since last purchase, as of census date)")
fig.show()
print(f"Recency: median {report['recency_median_days']:.0f} days, std {report['recency_std_days']:.0f} days")

Recency: median 218 days, std 153 days


In [4]:
fig = px.histogram(customer_summary, x="total_revenue", nbins=80,
                    title="Customer total revenue distribution (log y-axis)", log_y=True)
fig.show()
print(f"Monetary: median R$ {report['monetary_median']:.2f}, mean R$ {report['monetary_mean']:.2f}, "
      f"skew {report['monetary_skew']:.1f}")
print(f"P75 = R$ {report['monetary_p75']:.2f}, P90 = R$ {report['monetary_p90']:.2f}")
print(f"Revenue share of top 10% of customers by revenue: {report['top10pct_revenue_share']*100:.1f}%")

Monetary: median R$ 89.73, mean R$ 141.62, skew 9.7
P75 = R$ 154.74, P90 = R$ 279.99
Revenue share of top 10% of customers by revenue: 41.1%


**Recency and Monetary do have real variation** — recency spans the full ~2-year window (median 218 days, std 153), and monetary value is heavily right-skewed (skew ≈ 9.7; mean R$141.62 vs. median R$89.73), with the top 10% of customers contributing 41.1% of revenue. Those two dimensions are usable. Frequency is not.

**Decision: traditional RFM is not used.** Forcing a Frequency quintile onto a variable where 97% of observations share the same value would produce arbitrary, uninformative segment boundaries — exactly the kind of complexity-for-its-own-sake this project should avoid. Instead, Recency and Monetary are combined directly with a simple repeat/one-time behavioral split (see "Segmentation" below), which uses the two dimensions that actually separate customers.

## Retention Windows (Eligibility-Adjusted)

A customer whose first purchase was only 20 days before the census date cannot be observed returning within a 90-day window — including them in that denominator would understate retention. The rates below only count customers who have had *at least* the full window to return.

In [5]:
window_results = pd.DataFrame([seg.retention_window_rate(orders_analytics, w) for w in [30, 60, 90, 180]])
window_results["retention_rate_pct"] = (window_results["retention_rate"] * 100).round(2)
window_results

,window_days,n_eligible,n_returned,retention_rate,retention_rate_pct
0,30,86772,1384,0.015950,1.59
1,60,81203,1595,0.019642,1.96
2,90,75320,1719,0.022823,2.28
3,180,55907,1742,0.031159,3.12


In [6]:
fig = px.bar(window_results, x="window_days", y="retention_rate_pct",
             title="Eligibility-adjusted repeat-purchase rate by window",
             labels={"window_days": "window (days)", "retention_rate_pct": "retention rate (%)"},
             text_auto=".2f")
fig.show()

**Finding:** eligibility-adjusted retention rates (1.6% at 30 days, rising to 3.1% at 180 days) are all lower than the lifetime repeat-customer rate from Phase 4 (3.0%) except the 180-day figure, which nearly matches it — most customers who ever return do so within roughly 6 months. These numbers are stricter and more honest than a naive "% of all customers who returned within N days," which would be biased by customers who simply haven't had N days to return yet.

## Cohort Analysis

In [7]:
matrix, cohort_sizes = seg.build_cohort_retention_matrix(orders_analytics)
print(cohort_sizes.to_string(index=False))

cohort_month  cohort_size
     2016-09            1
     2016-10          262
     2016-12            1
     2017-01          717
     2017-02         1628
     2017-03         2503
     2017-04         2256
     2017-05         3451
     2017-06         3037
     2017-07         3752
     2017-08         4057
     2017-09         4004
     2017-10         4328
     2017-11         7060
     2017-12         5338
     2018-01         6842
     2018-02         6288
     2018-03         6774
     2018-04         6582
     2018-05         6506
     2018-06         5878
     2018-07         5949
     2018-08         6144


In [8]:
# limit to cohorts with a meaningful size and at least a few observed periods for readability
display_matrix = matrix.loc[matrix.index >= "2017-01"]
fig = px.imshow(display_matrix.values * 100, x=[f"M{c}" for c in display_matrix.columns], y=display_matrix.index,
                 color_continuous_scale="Blues", aspect="auto", text_auto=".1f",
                 labels={"color": "retention %"}, title="Cohort retention heatmap (% of cohort active in month M)")
fig.update_layout(height=600)
fig.show()

**Reading this heatmap:** each row is a cohort of customers who first purchased in that month; each column is how many months later. Blank (white/no-value) cells are **not zero** — they are calendar months that haven't happened yet for that cohort as of the dataset's end (2018-08), and are deliberately left out rather than implied to be churn. M0 is always 100% by construction (every customer is "active" in their own first month).

**Finding:** retention drops off immediately after M0 for every cohort — typically under 1% of a cohort returns in any single subsequent month, consistent with the very low overall repeat rate. There's no visually obvious cohort-over-cohort improvement or decline; low repeat purchasing looks like a structural feature of this customer base across the whole observation window, not a trend that's getting better or worse.

**Caution comparing cohorts directly:** a 2017-01 cohort has been observable for ~19 months, while a 2018-06 cohort has only had 2-3 months to generate a repeat purchase. Do not compare a late cohort's low *total* observed retention to an early cohort's — compare same-period columns (e.g. M1 vs M1) instead, which the heatmap already aligns for.

## Time to Second Purchase

In [9]:
t2p = seg.time_to_second_purchase(orders_analytics)
print(t2p.describe())
print()
print("Quartiles:", t2p.quantile([0.25, 0.5, 0.75]).to_dict())
for w in [30, 60, 90, 180]:
    print(f"Share of repeat customers who returned within {w} days (of those who DID return): {(t2p <= w).mean()*100:.1f}%")

count    2801.000000
mean       80.844698
std       109.899443
min         0.000000
25%         0.000000
50%        28.000000
75%       126.000000
max       608.000000
Name: days_to_second_purchase, dtype: float64

Quartiles: {0.25: 0.0, 0.5: 28.0, 0.75: 126.0}
Share of repeat customers who returned within 30 days (of those who DID return): 50.8%
Share of repeat customers who returned within 60 days (of those who DID return): 61.3%
Share of repeat customers who returned within 90 days (of those who DID return): 68.3%
Share of repeat customers who returned within 180 days (of those who DID return): 82.9%


In [10]:
fig = px.histogram(t2p, nbins=60, title="Days to second delivered order (repeat customers only)")
fig.update_layout(xaxis_title="days to second order", showlegend=False)
fig.show()

**This is a different statistic from the retention windows above** — it's conditional on already having returned (n = 2,801 repeat customers), describing *how fast* repeat customers come back, not *what share* of all customers do. Median time to a second order is 28 days, but the distribution is strongly right-skewed (mean 80.8 days), and a quarter of repeat customers placed their second order the very same day as their first (multi-order same-day checkouts).

**Right-censoring caveat:** a customer whose first purchase was very close to the dataset's end (2018-08-29) has had little time to make a second purchase, so this histogram inevitably undercounts recent customers who *would* have returned given more time. This is the same reason the eligibility-adjusted retention windows above exist — they correct for this bias; this raw time-to-second-purchase distribution does not (it only describes customers who already returned, which is well-defined regardless of censoring).

## Customer Value

In [11]:
print(customer_summary[["delivered_order_count", "total_revenue", "average_order_value", "recency_days", "tenure_days"]].describe())

       delivered_order_count  total_revenue  average_order_value  \
count           93358.000000   93358.000000         93358.000000   
mean                1.033420     141.621480           137.508262   
std                 0.209097     215.694014           209.860279   
min                 1.000000       0.850000             0.850000   
25%                 1.000000      47.650000            46.000000   
50%                 1.000000      89.730000            86.990000   
75%                 1.000000     154.737500           149.900000   
max                15.000000   13440.000000         13440.000000   

       recency_days   tenure_days  
count  93358.000000  93358.000000  
mean     236.941773      2.634032  
std      152.591453     24.955822  
min        0.000000      0.000000  
25%      113.000000      0.000000  
50%      218.000000      0.000000  
75%      345.000000      0.000000  
max      713.000000    633.000000  


In [12]:
value_by_repeat = (
    customer_summary.assign(is_repeat=customer_summary["delivered_order_count"] >= 2)
    .groupby("is_repeat")
    .agg(customers=("customer_unique_id", "count"),
         avg_total_revenue=("total_revenue", "mean"),
         avg_order_value=("average_order_value", "mean"))
)
value_by_repeat.index = value_by_repeat.index.map({True: "repeat", False: "one-time"})
value_by_repeat.round(2)

,customers,avg_total_revenue,avg_order_value
is_repeat,,,
one-time,90557,137.96,137.96
repeat,2801,260.05,122.96


**Finding:** repeat customers have ~1.9x the *observed total revenue* of one-time customers (R$260 vs. R$138 on average) simply from ordering more often — but their *per-order* AOV is actually slightly lower (R$123 vs. R$138), not higher. Repeat customers are not spending more per transaction; they're just transacting more often. These are "observed customer revenue" figures within the ~2-year dataset window, not a projected customer lifetime value.

In [13]:
state_repeat = (
    customer_summary.assign(is_repeat=customer_summary["delivered_order_count"] >= 2)
    .groupby("customer_state")
    .agg(customers=("customer_unique_id", "count"), repeat_customers=("is_repeat", "sum"))
)
state_repeat["repeat_rate_pct"] = (state_repeat["repeat_customers"] / state_repeat["customers"] * 100).round(2)
state_repeat.sort_values("customers", ascending=False).head(10)

,customers,repeat_customers,repeat_rate_pct
customer_state,,,
SP,39139,1213,3.10
RJ,11914,391,3.28
MG,10999,325,2.95
RS,5167,158,3.06
PR,4769,142,2.98
SC,3445,93,2.70
BA,3158,90,2.85
DF,2019,61,3.02
ES,1928,57,2.96


**Finding:** repeat rate by state is fairly uniform among the high-volume states (roughly 2.7–3.3% for the top 10 by customer count) — repeat purchasing does not appear to be concentrated in a particular region; it's a broad, low-magnitude pattern across the customer base rather than a geography-specific one.

## Segmentation

Behavior-based, not RFM (see "RFM Evaluation" above). Two documented, data-driven thresholds: **high value** = total revenue above the 75th percentile of all customers (R$154.74), and **recent** (for one-time customers only) = purchased within 90 days of the census date — the same window used in the retention analysis above, for consistency.

In [14]:
segmented = seg.assign_customer_segment(customer_summary)
print(f"Value threshold (P75 total_revenue): R$ {segmented.attrs['value_threshold']:.2f}")

segment_summary = (
    segmented.groupby("segment")
    .agg(customers=("customer_unique_id", "count"), total_revenue=("total_revenue", "sum"),
         avg_revenue=("total_revenue", "mean"))
    .sort_values("total_revenue", ascending=False)
)
segment_summary["pct_of_customers"] = (segment_summary["customers"] / len(segmented) * 100).round(1)
segment_summary["pct_of_revenue"] = (segment_summary["total_revenue"] / segment_summary["total_revenue"].sum() * 100).round(1)

assert segmented["segment"].isna().sum() == 0
assert abs(segment_summary["total_revenue"].sum() - m.total_revenue(orders_analytics)) < 0.01

segment_summary.round(2)

Value threshold (P75 total_revenue): R$ 154.74


,customers,total_revenue,avg_revenue,pct_of_customers,pct_of_revenue
segment,,,,,
High-Value One-Time,21661,7647707.17,353.06,23.2,57.8
One-Time Lapsed,55164,3874283.50,70.23,59.1,29.3
One-Time Recent,13732,971098.69,70.72,14.7,7.3
High-Value Repeat,1679,619710.07,369.09,1.8,4.7
Repeat Customer,1122,108698.68,96.88,1.2,0.8


In [15]:
fig = px.bar(segment_summary.reset_index(), x="segment", y="customers",
             title="Customers per segment", text_auto=True)
fig.show()

fig2 = px.bar(segment_summary.reset_index(), x="segment", y="pct_of_revenue",
              title="Share of total revenue per segment", text_auto=".1f")
fig2.show()

**Finding:** the segmentation splits a superficially uniform customer base ("97% buy once") into groups with very different value. `High-Value One-Time` customers — only 23% of customers — contribute 57.8% of total revenue, more than double any other segment, while `One-Time Lapsed` is the largest group by count (59%) but only 29.3% of revenue. The two repeat-customer segments together are just 3.0% of customers (matching Phase 4) and under 5.5% of revenue. This is a genuinely useful split for prioritization even without a repeat-purchase behavior to target: it separates "high-value customers worth a follow-up offer" from "low-value one-time buyers" regardless of whether they ever return.

## Analytical Limitations

- **Right-censoring** affects both cohort retention (recent cohorts have had less time to show repeat purchases) and time-to-second-purchase (recent first-time buyers haven't had time to return yet). Retention-window rates correct for this by restricting the denominator to eligible customers; the cohort matrix corrects for it by leaving unobservable cells blank rather than 0%; the raw time-to-second-purchase distribution does **not** correct for it (see that section).
- **RFM was deliberately not used.** This is a data-driven decision (97% Frequency=1), not a shortcut — see "RFM Evaluation" above.
- The census date (2018-08-29) is the dataset's own cutoff, not "today." All recency, tenure, and retention figures are relative to that fixed point, not a live system.
- "Observed customer revenue" describes revenue within the dataset window only — it is not a lifetime value projection, and no assumption is made about future purchasing.
- A minor (~0.06% of customers) boundary discrepancy exists between this notebook's Python recency calculation and the equivalent SQL query (`sql/06_customer_segmentation.sql`) at exactly the 90-day threshold, due to pandas' exact-elapsed-time day count vs. DuckDB's calendar-day `DATE_DIFF` — both are internally consistent, but customers with recency very close to 90 days may land in `One-Time Recent` in one implementation and `One-Time Lapsed` in the other. This does not affect any headline finding above.